In [1]:
import numpy as np
import pandas as pd

from sklearn.base import is_classifier
from sklearn.preprocessing import LabelBinarizer

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [2]:
class MLP1(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.2),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),

            nn.Linear(64, output_dim)
        )

    def forward(self, x):
        return self.network(x)

In [3]:
class TorchMLP1:
    def __init__(self, input_dim, output_dim, lr=5e-4):

        self.model = MLP1(input_dim, output_dim)

        self.optimizer = torch.optim.Adam(
            self.model.parameters(),
            lr=lr
        )

        self.criterion = nn.HuberLoss(delta=1.0)
    
    def predict(self, X, batch_size=1024):
        self.model.eval()

        X = torch.tensor(X, dtype=torch.float32)

        loader = torch.utils.data.DataLoader(
            X,
            batch_size=batch_size,
            shuffle=False
        )

        preds = []

        with torch.no_grad():
            for xb in loader:
                preds.append(self.model(xb).cpu())

        preds = torch.cat(preds, dim=0).numpy()
        return preds

In [4]:
def build_mlp1(input_dim, output_dim):
    return TorchMLP1(input_dim, output_dim)

In [5]:
def train_model1(
    wrapper,
    X_train,
    y_train,
    X_val=None,
    y_val=None,
    epochs=50,
    batch_size=32,
    verbose=False
):

    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=True
    )

    if X_val is not None:
        X_val = torch.tensor(X_val, dtype=torch.float32)
        y_val = torch.tensor(y_val, dtype=torch.float32)

    model = wrapper.model
    optimizer = wrapper.optimizer
    criterion = wrapper.criterion

    for epoch in range(epochs):

        model.train()
        train_loss = 0.0

        for xb, yb in train_loader:
            optimizer.zero_grad()

            preds = model(xb)
            loss = criterion(preds, yb)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # ---- validation ----
        if X_val is not None:
            model.eval()
            with torch.no_grad():
                val_preds = model(X_val)
                val_loss = criterion(val_preds, y_val).item()
        else:
            val_loss = None
        
        if verbose and (epoch + 1) % 10 == 0:
            msg = f"Epoch {epoch+1}: train={train_loss:.4f}"
            if val_loss is not None:
                msg += f", val={val_loss:.4f}"
            print(msg)

    return wrapper

In [6]:
# Code for Data
# Set seed
np.random.seed(1234)

n = 500
p = 7
rho = 0
# Create covariance matrix
s = np.full((p, p), rho)
np.fill_diagonal(s, 1.0)

# Generate multivariate normal data
x = np.random.multivariate_normal(
    mean=np.zeros(p),
    cov=s,
    size=n
)

# Convert to DataFrame with column names V1, V2, ...
x_df = pd.DataFrame(x, columns=[f"V{i}" for i in range(1, p+1)])

# Generate noise
noise = np.random.normal(loc=0, scale=0.1, size=n)


# Compute y
y1 = 4*x_df["V1"] + 3*x_df["V2"] + 2*x_df["V3"] + x_df["V4"] + noise
y2 = 4*x_df["V5"] + 3*x_df["V6"] + 2*x_df["V3"] + x_df["V4"] + noise
# Coefficient Important of: 4, 3, 4, 2, 4, 3, 0. See below!

y_df = pd.DataFrame({'y1': y1, 'y2': y2})

# Add another response variable
#noise1 = np.random.normal(loc=0, scale=0.1, size=n)
#y3 = noise1
#y_df = pd.DataFrame({'y1': y1, 'y2': y2, 'y3': y3})

X_df = pd.DataFrame(x_df)
#y_ser = pd.Series(y)
Y_df = pd.DataFrame(y_df)

feature_names = X_df.columns.tolist()

X_np = X_df.to_numpy(dtype=np.float32)
Y_np = Y_df.to_numpy(dtype=np.float32)

n, p = X_np.shape
n, yp = Y_np.shape

from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X_np,
    Y_np,
    test_size=0.2,
    random_state=123
)

In [7]:
model = build_mlp1(p, yp)
model = train_model1(
                model,
                X_train = X_train,
                y_train = Y_train,
                X_val = X_test,
                y_val = Y_test,
                epochs=100,
                batch_size=64,
                verbose=True
            )

Epoch 10: train=7.4151, val=0.7334
Epoch 20: train=5.6240, val=0.1988
Epoch 30: train=4.7621, val=0.2021
Epoch 40: train=5.5127, val=0.1464
Epoch 50: train=4.7963, val=0.1265
Epoch 60: train=5.0292, val=0.1484
Epoch 70: train=4.1944, val=0.1793
Epoch 80: train=4.2833, val=0.1402
Epoch 90: train=3.7287, val=0.1163
Epoch 100: train=4.1347, val=0.1722


In [8]:
from clique import clique, clique_eval, build_mlp

In [9]:
res_cv1 = clique(
    X=X_train,
    Y=Y_train,
    model_builder=build_mlp,
    folds=5,
    nsim=25,
    quantile_grid=True,
    loss_function=None,
    class_loss=False,
    epochs=50,
    batch_size=64,
    random_state=123,
    verbose=True
)
res_cv1

Epoch 10: train=7.0147, val=1.0539
Epoch 20: train=4.2113, val=0.4936
Epoch 30: train=3.8477, val=0.1898
Epoch 40: train=2.5898, val=0.1260
Epoch 50: train=2.8450, val=0.0986
Epoch 10: train=6.8654, val=0.8210
Epoch 20: train=3.8277, val=0.3767
Epoch 30: train=2.9952, val=0.1895
Epoch 40: train=2.9416, val=0.1449
Epoch 50: train=2.6518, val=0.0939
Epoch 10: train=6.9332, val=0.9572
Epoch 20: train=4.4747, val=0.4462
Epoch 30: train=2.9456, val=0.1834
Epoch 40: train=2.5978, val=0.1289
Epoch 50: train=2.5146, val=0.1081
Epoch 10: train=6.8224, val=1.0515
Epoch 20: train=4.1137, val=0.4677
Epoch 30: train=2.9317, val=0.2002
Epoch 40: train=2.8871, val=0.1431
Epoch 50: train=2.7733, val=0.1162
Epoch 10: train=6.8238, val=1.0624
Epoch 20: train=4.1346, val=0.4793
Epoch 30: train=3.6142, val=0.1871
Epoch 40: train=2.6023, val=0.1700
Epoch 50: train=2.8389, val=0.1079
Finished variable 1/7
Finished variable 2/7
Finished variable 3/7
Finished variable 4/7
Finished variable 5/7
Finished variab

{'models': [<clique.TorchMLP at 0x2c3b2f5dcd0>,
 'local_imp':             0         1         2         3         4         5         6
 0    3.381416  2.533334  6.000834  2.046893  5.042500  2.748687  0.079813
 1    3.932152  2.327975  2.784119  4.952725  3.618666  2.505378 -0.117258
 2    9.286407  2.326923  2.603114  0.690218  2.933286  5.487059  0.107332
 3    2.993710  4.740947  6.865222  3.004814  3.301158  2.526881 -0.332355
 4    2.996483  2.304528  6.191612  0.839136  7.498435  1.743383  0.064656
 ..        ...       ...       ...       ...       ...       ...       ...
 395  4.808241  2.726922  2.474043  1.332856  3.350316  2.414470  0.045946
 396  4.215522  2.737617  3.465573  1.324623  3.640552  3.514576  0.024358
 397  3.271138  2.576123  3.334251  1.590177  3.712259  2.740725  0.048340
 398  5.537679  4.012433  4.489181  1.753716  3.387303  3.558701  0.120687
 399  3.277762  2.644510  3.451209  1.339092  4.343860  2.526733  0.104401
 
 [400 rows x 7 columns]}

In [10]:
res_cv1["local_imp"].mean(axis=0)
# Coefficient Important of: 4, 3, 4, 2, 4, 3, 0. See below!

0    4.276106
1    3.197858
2    3.954830
3    1.672218
4    4.325202
5    3.056006
6   -0.003786
dtype: float32

In [11]:
res_test1 = clique_eval(
    model=model,
    X_train=X_train,
    Y_train=Y_train,
    X_test=X_test,
    Y_test=Y_test
)
res_test1

,0,1,2,3,4,5,6
0,4.320542,2.519278,4.885870,0.963450,3.082681,3.233867,0.131535
1,3.005052,1.744937,3.173799,2.193017,3.776913,3.623968,-0.065211
2,11.682037,1.651857,2.630093,2.743745,3.425015,2.331996,-0.046258
3,4.200172,2.592252,4.820873,0.125951,2.763649,1.716374,-0.144273
4,5.168291,2.253145,3.274484,1.268382,4.212254,2.748324,-0.156102
...,...,...,...,...,...,...,...
95,4.354982,2.895801,3.887296,1.622145,3.135016,2.346176,0.012619
96,5.167026,1.692064,6.902397,0.889047,2.697709,2.524822,-0.028732
97,4.751371,2.652532,3.641925,2.282017,3.708615,5.219151,-0.047343
98,5.272708,2.698246,2.770744,1.963350,4.392159,4.975646,0.143542


In [12]:
res_test1.mean(axis=0)
# Coefficient Important of: 4, 3, 4, 2, 4, 3, 0. See below!

0    4.387186
1    2.993275
2    3.506616
3    1.529260
4    4.235521
5    3.074678
6   -0.020335
dtype: float32

In [13]:
res_train1 = clique_eval(
    model=model,
    X_train=X_train,
    Y_train=Y_train
)
res_train1

,0,1,2,3,4,5,6
0,3.346919,2.389484,5.201634,1.218181,4.461801,2.279922,0.063572
1,3.811032,1.720193,1.724883,4.362063,3.053262,2.233607,-0.019368
2,9.816972,2.196107,2.835867,0.710614,2.905357,6.037951,0.267137
3,3.367391,4.009200,7.549948,2.975530,3.386454,2.455912,0.113988
4,3.831587,2.615179,6.737429,0.986942,7.481119,2.205841,0.076518
...,...,...,...,...,...,...,...
395,5.006638,2.958608,3.124765,1.670817,3.513935,2.563409,0.147501
396,4.138841,2.673625,3.189918,1.005946,3.292854,3.224255,0.108388
397,3.171116,2.436411,2.723429,1.566970,3.318007,2.673894,0.135108
398,5.324025,3.679972,4.626338,0.998588,2.892142,2.538309,0.133484


In [14]:
res_train1.mean(axis=0)
# Coefficient Important of: 4, 3, 4, 2, 4, 3, 0. See below!

0    4.271908
1    3.179694
2    3.872743
3    1.502408
4    4.194614
5    2.994653
6    0.029487
dtype: float32